# Day 06: Generative Modeling — 1D VAE

Welcome to practical 6! Implement the ELBO and train a variational autoencoder on 1-D synthetic data.

This notebook follows the MIT lab style: abstract base classes, PyTorch, and LaTeX in markdown. Wherever you see a `# YOUR CODE HERE` marker, replace the `raise NotImplementedError(...)` below it with your own implementation.

In [ ]:
import math
from abc import ABC, abstractmethod
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

device = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

The ELBO for a VAE is $\mathcal{L} = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{\mathrm{KL}}(q_\phi(z|x) \| p(z))$. For Gaussian $q=\mathcal{N}(\mu,\mathrm{diag}(\sigma^2))$ and $p=\mathcal{N}(0,I)$, $D_{\mathrm{KL}} = \tfrac{1}{2}\sum_j (\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1)$.

In [ ]:
# 1-D mixture of Gaussians as "data"
data = torch.cat([
    torch.randn(500) * 0.3 + 2.0,
    torch.randn(500) * 0.3 - 2.0,
]).unsqueeze(1)
loader = DataLoader(data, batch_size=64, shuffle=True)

### Exercise 6.1: KL divergence

**Your job**: Implement `kl_gaussian` for diagonal Gaussians vs standard normal.

In [ ]:
def kl_gaussian(mu, logvar):
    # YOUR CODE HERE
    raise NotImplementedError("TODO: complete this exercise")

mu = torch.randn(8, 4)
logvar = torch.randn(8, 4)
print("KL (sample):", kl_gaussian(mu, logvar).item())

### Exercise 6.2: VAE architecture

**Your job**: Build encoder and decoder MLPs.

In [ ]:
class VAE1D(nn.Module):
    def __init__(self, latent=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU())
        self.mu = nn.Linear(32, latent)
        self.logvar = nn.Linear(32, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, 32), nn.ReLU(), nn.Linear(32, 1))

    def reparameterize(self, mu, logvar):
        # YOUR CODE HERE
        raise NotImplementedError("TODO: complete this exercise")

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

vae = VAE1D().to(device)

### Exercise 6.3: ELBO loss

**Your job**: Combine reconstruction MSE and KL term.

In [ ]:
def elbo_loss(recon, x, mu, logvar):
    # YOUR CODE HERE
    raise NotImplementedError("TODO: complete this exercise")

opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
for epoch in range(20):
    vae.train()
    total = 0.0
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        recon, mu, logvar = vae(batch)
        loss, _, _ = elbo_loss(recon, batch, mu, logvar)
        loss.backward()
        opt.step()
        total += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}: ELBO loss {total/len(loader):.4f}")

### Exercise 6.4: Sample from the prior

**Your job**: Decode $z \sim \mathcal{N}(0,I)$ and histogram.

In [ ]:
@torch.no_grad()
def sample_vae(model, n=500):
    z = torch.randn(n, 2)
    return model.decoder(z).squeeze().numpy()

samples = sample_vae(vae)
plt.hist(data.numpy().ravel(), bins=40, alpha=0.5, label="data", density=True)
plt.hist(samples, bins=40, alpha=0.5, label="VAE samples", density=True)
plt.legend()
plt.title("1D VAE: data vs samples")
plt.show()

## Reflection (Day 6)

Take a few minutes to answer in your own words:

1. What was the most important concept you practiced today (VAEs and the ELBO)?
2. Where did you get stuck, and how did you resolve it?
3. How would you explain today's material to a classmate in two sentences?
4. What would you like to explore further?